# 繪師 (PAINTER SI) — Colab ComfyUI + BackendGateway 三神模啟動器

- 白名單信箱：`qseft951753@gmail.com`
- 內建三大頂尖底模自動下載（已存在則秒速跳過）：
  1. **FLUX.1-dev FP8**（寫實人像 / 風景天花板，17.2 GB）
  2. **RealVisXL V5.0**（SDXL 最強寫實攝影，相容寫實 LoRA，6.9 GB）
  3. **Animagine XL 3.1**（動漫 / 光影 / 遊戲 / 插畫天花板，6.9 GB）

In [ ]:
# @title 1. 掛載 Google Drive 與快速定位「繪師」目錄
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

candidates = [
    '/content/drive/MyDrive/麻糬紅豆湯/繪師',
    '/content/drive/MyDrive/繪師',
]
HUISHI_DIR = next((p for p in candidates if os.path.exists(os.path.join(p, 'backend_gateway.py'))), None)

if not HUISHI_DIR:
    root_drive = '/content/drive/MyDrive'
    for cur_root, dirs, files in os.walk(root_drive):
        depth = cur_root.replace(root_drive, '').count(os.sep)
        if 'backend_gateway.py' in files:
            HUISHI_DIR = cur_root
            break
        if depth >= 3:
            dirs[:] = []

if not HUISHI_DIR:
    raise FileNotFoundError('❌ 在 Google Drive 找不到 backend_gateway.py')

BASE_DIR = f'{HUISHI_DIR}/AI_Studio'
for sub in ['models/checkpoints', 'models/loras', 'bgm', 'outputs', 'config']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

print('✅ 已定位繪師資料夾:', HUISHI_DIR)
print('✅ AI_Studio 目錄已就緒:', BASE_DIR)

In [ ]:
# @title 2. 安裝相依套件、FFmpeg 與 cloudflared
!pip install -q fastapi uvicorn firebase-admin pydantic pillow
!apt-get -qq update && apt-get -qq install -y ffmpeg
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print('✅ 套件、FFmpeg 與 cloudflared 安裝完成')

In [ ]:
# @title 3. 安裝並啟動 ComfyUI（自動下載 FLUX.1-dev + RealVisXL V5.0 + Animagine XL 3.1）
import os, subprocess, time, urllib.request

CKPT_DIR = f'{BASE_DIR}/models/checkpoints'

if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt

extra_yaml = f'''comfyui:
    base_path: {BASE_DIR}/models
    checkpoints: checkpoints
    loras: loras
'''
with open('/content/ComfyUI/extra_model_paths.yaml', 'w') as f:
    f.write(extra_yaml)

FLAGSHIP_MODELS = [
    ('animagine-xl-3.1.safetensors', 'https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors'),
    ('RealVisXL_V5.0_fp16.safetensors', 'https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors'),
    ('flux1-dev-fp8.safetensors', 'https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors'),
]

for fname, url in FLAGSHIP_MODELS:
    fpath = os.path.join(CKPT_DIR, fname)
    if not os.path.exists(fpath):
        print(f'⏳ 首次下載頂尖底模：{fname}（之後直接重複使用）...')
        !wget -q --show-progress -O "{fpath}" "{url}"
    else:
        print(f'✅ 已就緒：{fname}')

comfy_proc = subprocess.Popen(
    ['python3', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'],
    stdout=open('/content/comfyui.log', 'w'),
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(45):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(2)

if not ready:
    raise RuntimeError('❌ ComfyUI 啟動逾時，請查看 /content/comfyui.log')
print('✅ ComfyUI API 已就緒！三大神模（FLUX.1-dev + RealVisXL V5.0 + Animagine XL 3.1）已載入！')

In [ ]:
# @title 4. 載入 BackendGateway 並啟動 Cloudflare Tunnel
import os, sys, shutil, re, subprocess, threading, time
from datetime import datetime
from zoneinfo import ZoneInfo
import uvicorn
from firebase_admin import firestore

os.environ['HUISHI_ALLOWED_EMAIL'] = 'qseft951753@gmail.com'
os.environ['FIREBASE_SERVICE_ACCOUNT_JSON'] = f'{BASE_DIR}/config/serviceAccountKey.json'
os.environ['HUISHI_OUTPUT_DIR'] = f'{BASE_DIR}/outputs'
os.environ['HUISHI_BGM_DIR'] = f'{BASE_DIR}/bgm'
os.environ['COMFYUI_URL'] = 'http://127.0.0.1:8188'

for fname in ['backend_gateway.py', 'comfy_client.py', 'ffmpeg_composer.py']:
    src = os.path.join(HUISHI_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, f'/content/{fname}')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import backend_gateway
backend_gateway.init_firebase_admin()

server_thread = threading.Thread(
    target=uvicorn.run,
    args=(backend_gateway.app,),
    kwargs={'host': '0.0.0.0', 'port': 8000, 'log_level': 'warning'},
    daemon=True,
)
server_thread.start()
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
for line in tunnel_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

if not tunnel_url:
    raise RuntimeError('❌ 無法取得 Cloudflare Tunnel URL')

gpu_info = backend_gateway.detect_gpu_vram()
db = firestore.client()
db.collection('system').document('gateway').set({
    'url': tunnel_url,
    'status': 'online',
    'updatedAt': datetime.now(ZoneInfo('Asia/Taipei')).isoformat(),
    'gpuName': gpu_info['gpu_name'],
    'vramGb': gpu_info['vram_gb'],
    'vramMode': gpu_info['vram_mode'],
    'workflowTier': gpu_info['workflow_tier'],
})

print('🚀 BackendGateway + ComfyUI 三神模已全線就緒！')
print('🔗 Tunnel URL:', tunnel_url)
print('🖥️ GPU 資訊:', gpu_info)

tunnel_proc.wait()